In [1]:
!pip install google-adk python-dotenv requests google-genai litellm

In [ ]:
import os
from google.colab import userdata
import logging
from google.adk.sessions import InMemorySessionService
from google.adk.models import LlmRequest, LlmResponse
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm
from google.genai import types
from typing import Optional
logging.getLogger("google_genai.types").setLevel(logging.ERROR)


# Required for Gemini and Geocoding
os.environ["GOOGLE_API_KEY"] = "GOOGLE_API_KEY"

# OpenAI (if using GPT models)
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"

# Anthropic (if using Claude)
os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_KEY"

GOOGLE_MAPS_API_KEY = "GOOGLE_API_KEY"

In [19]:
import requests
from typing import Dict

def get_coordinates(location_name: str) -> Dict[str, float | str]:
    """
    Converts a place name (city, address, etc.) into latitude and longitude.

    Uses the Google Maps Geocoding API to resolve the location. Returns
    an error key if the location cannot be found or the API returns a
    non-OK status.

    Args:
        location_name (str): The name of the city or place to geocode.

    Returns:
        Dict[str, Union[float, str]]: On success, a dictionary with:
            - 'lat' (float): Latitude of the location.
            - 'lon' (float): Longitude of the location.
            On failure, a dictionary with:
            - 'error' (str): A human-readable error message.

    Raises:
        requests.exceptions.RequestException: If the Geocoding API is
            unreachable or the request fails.
    """
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={location_name}&key={GOOGLE_MAPS_API_KEY}"
    response = requests.get(url).json()

    if response.get("status") == "OK":
        location = response["results"][0]["geometry"]["location"]
        return {"lat": location["lat"], "lon": location["lng"]}
    return {"error": "Location not found"}

def get_nws_weather(lat: float, lon: float) -> str:
    """
    Retrieves the current weather forecast from the National Weather Service (NWS).

    Args:
        lat (float): Latitude coordinate.
        lon (float): Longitude coordinate.

    Returns:
        str: A human-readable forecast string including period name,
             detailed forecast description, and current temperature.
             Returns an error message string if the location is unsupported.

     Raises:
        requests.exceptions.RequestException: If the NWS API is unreachable.
    """
    # NWS requires a User-Agent header (use your own email/app name)
    headers = {"User-Agent": "WeatherAlertAgent/1.0 (contact@example.com)", "Accept": "application/geo+json"}

    # Step 1: Get grid point metadata from coordinates
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    res = requests.get(points_url, headers=headers).json()

    if "properties" not in res:
        return "Weather data unavailable for this location (NWS is US only)."

    # Step 2: Get the actual forecast from the grid endpoint provided in properties
    forecast_url = res["properties"]["forecast"]
    forecast_res = requests.get(forecast_url, headers=headers).json()

    # Get the current period's forecast summary
    current = forecast_res["properties"]["periods"][0]
    return (f"Forecast for {current['name']}: {current['detailedForecast']} "
            f"Temperature: {current['temperature']}°{current['temperatureUnit']}.")

In [20]:
from google.adk.agents import LlmAgent
from google.adk.models.lite_llm import LiteLlm

def create_weather_agent(model_choice="gemini-flash-lite-latest"):
    """
    Creates a weather agent. Pass a Gemini model name as a string,
    or a LiteLlm() object for third-party models.
    """
    agent_instructions = """
    You are a weather analyst.
    1. Use 'get_coordinates' to find the lat/lon of a user's location.
    2. Use 'get_nws_weather' with those coordinates to get the forecast.
    3. Provide a clear summary. Note that NWS only works for US locations.
    """

    return LlmAgent(
        name="weather_agent",
        model=model_choice,        # accepts string for Gemini, or LiteLlm() for others
        instruction=agent_instructions,
        tools=[get_coordinates, get_nws_weather],
    )


In [21]:
APP_NAME = "weather_app"

session_service = InMemorySessionService()

# --- Option A: Gemini (no change) ---
weather_agent = create_weather_agent("gemini-flash-lite-latest")

# --- Option B: OpenAI GPT-4o ---
#weather_agent = create_weather_agent(LiteLlm(model="openai/gpt-4o"))

# --- Option C: Anthropic Claude ---
#weather_agent = create_weather_agent(LiteLlm(model="anthropic/claude-3-5-haiku-20241022"))

runner = Runner(
    agent=weather_agent,
    app_name=APP_NAME,
    session_service=session_service,
)


In [22]:
async def chat_once(user_id: str, session_id: str, message: str) -> str:
    """
    Send one message to the agent and return its final response text.
    """
    user_content = types.Content(
        parts=[types.Part(text=message)],
        role="user",
    )

    final_text = ""

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=user_content,
    ):
        # The ADK emits events; we only care about the final response.
        if event.is_final_response() and event.content and event.content.parts:
            final_text = event.content.parts[0].text

    return final_text




In [23]:
APP_NAME = "weather_app"
USER_ID = "demo_user"
SESSION_ID = "session_1"

session_service = InMemorySessionService()
weather_agent = create_weather_agent()
runner = Runner(
    agent=weather_agent,
    app_name=APP_NAME,
    session_service=session_service,
)

async def setup():
    session = await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )
    print(f"Session created: {session.id}")

await setup()


Session created: session_1


In [24]:
## Run only if you get a 404 error for the currently used Gemini model to get a list of available models
import google.generativeai as genai

# Configure the API key (already done in your setup, but good to ensure)
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Available Gemini models:")
for m in genai.list_models():
    # Filter for models that support 'generateContent' and are Gemini models
    if "generateContent" in m.supported_generation_methods and "gemini" in m.name:
        print(f"- {m.name}")


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available Gemini models:
- models/gemini-2.5-flash
- models/gemini-2.5-pro
- models/gemini-2.0-flash
- models/gemini-2.0-flash-001
- models/gemini-2.0-flash-exp-image-generation
- models/gemini-2.0-flash-lite-001
- models/gemini-2.0-flash-lite
- models/gemini-2.5-flash-preview-tts
- models/gemini-2.5-pro-preview-tts
- models/gemini-flash-latest
- models/gemini-flash-lite-latest
- models/gemini-pro-latest
- models/gemini-2.5-flash-lite
- models/gemini-2.5-flash-image
- models/gemini-2.5-flash-lite-preview-09-2025
- models/gemini-3-pro-preview
- models/gemini-3-flash-preview
- models/gemini-3.1-pro-preview
- models/gemini-3.1-pro-preview-customtools
- models/gemini-3.1-flash-lite-preview
- models/gemini-3-pro-image-preview
- models/gemini-3.1-flash-image-preview
- models/gemini-robotics-er-1.5-preview
- models/gemini-2.5-computer-use-preview-10-2025


In [25]:
response = await chat_once(
    user_id=USER_ID,
    session_id=SESSION_ID,
    message="What is the weather like in Springfield, Illinois?",
)
print(response)


The current weather forecast for Springfield, Illinois is:

Tonight, there will be areas of fog before 1 AM, followed by a chance of rain showers between 1 AM and 3 AM, and then areas of fog with a chance of showers and thunderstorms. It will be mostly cloudy with a low around 52°F, rising to around 57°F overnight. The south-southeast wind will be 3 to 9 mph, with gusts as high as 16 mph. There is a 50% chance of precipitation, with less than a tenth of an inch of new rainfall possible. The current temperature is 52°F.


In [26]:
response = await chat_once(
    user_id=USER_ID,
    session_id=SESSION_ID,
    message="What is the weather like in Austin, Texas?",
)
print(response)

The current weather forecast for Austin, Texas is:

Tonight, it will be mostly cloudy with a low around 69°F. The south-southeast wind will be between 10 and 15 mph, with gusts as high as 30 mph. The current temperature is 69°F.


In [27]:
# Example list of user messages with the last one flagged by Model Armor
messages = [
    "What is the weather like in Springfield, Illinois?",
    "What about Chicago, Illinois?",
    "Now tell me the weather in New York City.",
]

results = []

for msg in messages:
    resp = await chat_once(
        user_id=USER_ID,
        session_id=SESSION_ID,
        message=msg,
    )
    print(f"\nUser: {msg}\nAgent: {resp}\n")
    results.append(resp)


User: What is the weather like in Springfield, Illinois?
Agent: The current weather forecast for Springfield, Illinois is:

Tonight, there will be areas of fog before 1 AM, followed by a chance of rain showers between 1 AM and 3 AM, and then areas of fog with a chance of showers and thunderstorms. It will be mostly cloudy with a low around 52°F, rising to around 57°F overnight. The south-southeast wind will be 3 to 9 mph, with gusts as high as 16 mph. There is a 50% chance of precipitation, with less than a tenth of an inch of new rainfall possible. The current temperature is 52°F.


User: What about Chicago, Illinois?
Agent: The current weather forecast for Chicago, Illinois is:

Tonight, there will be widespread fog. It will be cloudy with a low around 38°F. The east wind will be around 5 mph. The current temperature is 38°F.


User: Now tell me the weather in New York City.
Agent: The current weather forecast for New York City is:

Tonight, there will be rain. It will be cloudy wit